# NL2SH — Qwen2.5-Coder-3B-Instruct Supervised Fine-Tuning (SFT)

**Project:** NL2SH (Natural Language to Shell)  
**Base Model:** `Qwen/Qwen2.5-Coder-3B-Instruct`  
**Method:** Parameter-Efficient Fine-Tuning (PEFT) with LoRA / QLoRA (4-bit BitsAndBytes)  
**Dataset:** NL2SH Clean SFT Split (28,167 train, 3,129 validation records)  
**Target Hardware:** Google Colab Free / Pro (NVIDIA T4 16GB, L4 24GB, or A100 40GB)

---

### SFT Mapping Objective
- **Input:** Natural-language instruction describing a shell task.
- **Output:** Exact, executable single Bash command (zero markdown, zero commentary, zero conversational filler).
- **Holdout Safety:** The official 300-record benchmark (`alfa_test.csv`) is strictly held out and never accessed during training or validation.

## 1. Inspect Hardware & Auto-Detect GPU Environment

In [ ]:
# Inspect GPU type and available VRAM
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise SystemError("CUDA GPU not detected! In Google Colab, go to Runtime > Change runtime type > Select T4, L4, or A100 GPU.")

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"\nDetected GPU: {gpu_name} ({total_vram_gb:.2f} GB VRAM)")

# Adaptive config based on detected VRAM
MAX_SEQ_LENGTH = 384  # Guarantees 100% of NL2SH dataset fits with zero command truncation
if total_vram_gb >= 20.0:
    print("\n[Configuration] High-VRAM GPU (L4 / A100): Using 16-bit LoRA with batch size 8.")
    LOAD_IN_4BIT = False
    BATCH_SIZE = 8
    GRAD_ACCUM = 2
    COMPUTE_DTYPE = torch.bfloat16
elif total_vram_gb >= 14.0:
    print("\n[Configuration] Standard T4 / V100 (15-16 GB): Using 4-bit QLoRA (NF4) with batch size 4.")
    LOAD_IN_4BIT = True
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    COMPUTE_DTYPE = torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16
else:
    print("\n[Configuration] Conservative Fallback (< 14 GB): Using 4-bit QLoRA with batch size 2.")
    LOAD_IN_4BIT = True
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    COMPUTE_DTYPE = torch.float16
    MAX_SEQ_LENGTH = 256

## 2. Install SFT & PEFT Dependencies

In [ ]:
# Install pinned mutually-compatible training libraries
!pip install -q "transformers>=4.46.0,<4.49.0" "peft>=0.13.0,<0.15.0" "trl>=0.11.0,<0.13.0" "bitsandbytes>=0.44.0" "accelerate>=0.34.0,<1.0.0" "datasets>=2.21.0,<3.0.0" pyyaml


## 3. Clone NL2SH Repository & Verify Processed Datasets

In [ ]:
import os
from pathlib import Path

# Clone repository if running in fresh Colab session
if not Path("data/processed/train.jsonl").exists():
    !git clone https://github.com/gurupawan265/NL2SH.git
    os.chdir("NL2SH")

train_path = Path("data/processed/train.jsonl")
val_path = Path("data/processed/val.jsonl")

assert train_path.exists(), f"Training data missing: {train_path}"
assert val_path.exists(), f"Validation data missing: {val_path}"

train_lines = [l for l in train_path.read_text(encoding='utf-8').splitlines() if l.strip()]
val_lines = [l for l in val_path.read_text(encoding='utf-8').splitlines() if l.strip()]

print(f"Verified Clean Training Records:   {len(train_lines):,}")
print(f"Verified Clean Validation Records: {len(val_lines):,}")
print(f"Total SFT Dataset:                {len(train_lines) + len(val_lines):,}")

## 4. Setup Chat Template & Format SFT Datasets

Each record is converted into Qwen2.5 ChatML format:
- **System:** Instructs the model to output raw Bash command only.
- **User:** Natural-language instruction.
- **Assistant:** Ground-truth Bash command.

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an expert Linux shell assistant. Your task is to translate natural language "
    "instructions into exact, executable Bash commands. Output only the command with no markdown, "
    "explanation, or preamble."
)

def format_qwen_chat(instruction: str, command: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n{command.strip()}<|im_end|>"
    )

def load_dataset_from_jsonl(path):
    texts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                texts.append(format_qwen_chat(row['instruction'], row['command']))
    return Dataset.from_dict({"text": texts})

print("Formatting datasets into Qwen ChatML...")
train_dataset = load_dataset_from_jsonl("data/processed/train.jsonl")
val_dataset = load_dataset_from_jsonl("data/processed/val.jsonl")

print(f"Train dataset samples: {len(train_dataset):,}")
print(f"Val dataset samples:   {len(val_dataset):,}")
print("\n--- Sample ChatML Record ---")
print(train_dataset[0]["text"])

## 5. Load Base Model with QLoRA / LoRA Configuration

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

# 1. Tokenizer
print(f"Loading tokenizer: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Quantization
bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

# 3. Base Model
print(f"Loading base model: {MODEL_ID}...")
model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": COMPUTE_DTYPE,
    "device_map": "auto",
}
if bnb_config is not None:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()

# 4. PEFT LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
print("\n--- Trainable Parameters ---")
model.print_trainable_parameters()

## 6. Configure Training Arguments & SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import DataCollatorForCompletionOnlyLM, SFTTrainer

OUTPUT_DIR = "checkpoints/qwen2.5-coder-3b-nl2sh-sft"
ADAPTER_DIR = "adapters/qwen2.5-coder-3b-nl2sh-sft"

# Response-only loss masking: masks user/system prompt so loss is computed ONLY on bash command tokens
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    fp16=(COMPUTE_DTYPE == torch.float16),
    optim="paged_adamw_8bit" if LOAD_IN_4BIT else "adamw_torch",
    seed=42,
    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=collator,
    tokenizer=tokenizer,
    args=training_args,
)

print("SFT Trainer with DataCollatorForCompletionOnlyLM configured successfully.")


## 7. Run SFT Training

In [ ]:
# Execute fine-tuning
print("Starting Qwen2.5-Coder-3B SFT training...")
trainer.train()

# Save final PEFT adapter and tokenizer
print(f"\nSaving fine-tuned adapter to {ADAPTER_DIR}...")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Model training and adapter export complete!")

## 8. Inference Verification with Fine-Tuned Adapter

In [ ]:
def generate_command(instruction: str, max_new_tokens: int = 64) -> str:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.encode("<|im_end|>")[0],
        )
    raw_out = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return raw_out.strip()

test_queries = [
    "Find all '.jpg' files inside the directory /workspace/photos",
    "Search case-insensitively for 'failed' in /workspace/app.log",
    "Replace all occurrences of 'foo' with 'bar' in /workspace/sample.txt",
    "Set permissions on /workspace/private_dir to 700",
]

print("\n--- Inference Verification Sample Runs ---")
for q in test_queries:
    cmd = generate_command(q)
    print(f"Query:     {q}")
    print(f"Generated: {cmd}\n")

## 9. Export Adapter for Local Integration or Hugging Face Hub

In [ ]:
# Zip adapter directory for easy download from Google Colab
!zip -r qwen2.5_coder_3b_nl2sh_adapter.zip adapters/qwen2.5-coder-3b-nl2sh-sft/

# Download adapter directly in Colab browser
try:
    from google.colab import files
    files.download('qwen2.5_coder_3b_nl2sh_adapter.zip')
except ImportError:
    print("Not running in Google Colab; adapter saved locally in adapters/qwen2.5-coder-3b-nl2sh-sft/")